In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from tqdm import tqdm
import seaborn as sns
import math
from numpy.lib.stride_tricks import sliding_window_view

%matplotlib inline

In [ ]:
# MNIST: 28x28 grayscale, 10 digit classes, normalized to [-1, 1]
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))  # [0,1] -> [-1, 1]
])

train_dataset = datasets.MNIST('./data', train=True, download=True, transform=transform)
train_loader  = DataLoader(train_dataset, batch_size=256, shuffle=True, num_workers=2, drop_last=True)

print(f"Dataset size : {len(train_dataset):,}")
print(f"Image shape  : {train_dataset[0][0].shape}")

fig, axes = plt.subplots(2, 8, figsize=(16, 4))
for i, ax in enumerate(axes.flat):
    img, label = train_dataset[i]
    ax.imshow(img.squeeze(), cmap='gray', vmin=-1, vmax=1)
    ax.set_title(str(label), fontsize=8)
    ax.axis('off')
plt.suptitle('MNIST Training Samples')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# JiT Architecture
# "Back to Basics: Let Denoising Generative Models Denoise"
# https://arxiv.org/abs/2511.13720  (Li & He, MIT 2025)
# ============================================================
#
# Core insight: natural images lie on a low-dimensional manifold.
#   - epsilon/v-prediction targets OFF-manifold quantities -> fails at high patch dim
#   - x-prediction targets the ON-manifold clean image   -> works with bottleneck nets
#
# Architecture (JiT = Just image Transformers):
#   1. Patchify raw pixels (no tokenizer / VQ-VAE)
#   2. Bottleneck linear embed: patch_dim -> hidden_dim  (patch_dim CAN exceed hidden_dim)
#   3. Standard ViT blocks with AdaLN time+label conditioning
#   4. x-prediction head: hidden_dim -> patch_dim  (predicts clean image directly)
#   5. Classifier-Free Guidance for class-conditional generation


# ---- Sinusoidal time embedding --------------------------------
class SinusoidalPosEmb(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, t):
        # t: (B,)  scalar time in [0, 1]
        half  = self.dim // 2
        denom = max(half - 1, 1)
        freqs = torch.exp(
            -math.log(10000) * torch.arange(half, device=t.device).float() / denom
        )  # (half,)
        emb = t[:, None] * freqs[None, :]          # (B, half)
        return torch.cat([emb.sin(), emb.cos()], dim=-1)  # (B, dim)


# ---- Adaptive Layer Norm (AdaLN) ------------------------------
# Standard conditioning mechanism in DiT / JiT:
#   scale, shift = MLP(cond)
#   out = (1 + scale) * LayerNorm(x) + shift
class AdaLN(nn.Module):
    def __init__(self, hidden_dim, cond_dim):
        super().__init__()
        self.norm = nn.LayerNorm(hidden_dim, elementwise_affine=False)
        self.proj = nn.Sequential(
            nn.SiLU(),
            nn.Linear(cond_dim, 2 * hidden_dim)
        )
        # Zero-init: start as identity transform for stable training
        nn.init.zeros_(self.proj[-1].weight)
        nn.init.zeros_(self.proj[-1].bias)

    def forward(self, x, cond):
        # x:    (B, N, hidden_dim)
        # cond: (B, cond_dim)
        scale, shift = self.proj(cond).chunk(2, dim=-1)   # each (B, hidden_dim)
        return self.norm(x) * (1 + scale[:, None]) + shift[:, None]


# ---- Transformer Block ----------------------------------------
class TransformerBlock(nn.Module):
    def __init__(self, hidden_dim, num_heads, cond_dim, mlp_ratio=4):
        super().__init__()
        self.adaLN1 = AdaLN(hidden_dim, cond_dim)
        self.attn   = nn.MultiheadAttention(hidden_dim, num_heads, batch_first=True)
        self.adaLN2 = AdaLN(hidden_dim, cond_dim)
        self.ff = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim * mlp_ratio),
            nn.GELU(),
            nn.Linear(hidden_dim * mlp_ratio, hidden_dim),
        )

    def forward(self, x, cond):
        x_norm = self.adaLN1(x, cond)
        attn_out, _ = self.attn(x_norm, x_norm, x_norm)
        x = x + attn_out
        x = x + self.ff(self.adaLN2(x, cond))
        return x


# ---- JiT Model ------------------------------------------------
class JiT(nn.Module):
    """
    Just image Transformers (JiT)

    For MNIST (28x28, 1ch) with patch_size=7:
      patch_dim  = 7*7*1 = 49   <- high-dimensional patches
      n_patches  = (28/7)^2 = 16
      hidden_dim = 128          <- bottleneck: 49 -> 128 (mild)
      Try patch_size=14 (patch_dim=196) to stress-test the manifold hypothesis
    """
    def __init__(
        self,
        img_size     = 28,
        patch_size   = 7,
        channels     = 1,
        hidden_dim   = 128,
        depth        = 6,
        num_heads    = 4,
        num_classes  = 10,
        time_emb_dim = 128,
        mlp_ratio    = 4,
    ):
        super().__init__()
        self.img_size    = img_size
        self.patch_size  = patch_size
        self.channels    = channels
        self.hidden_dim  = hidden_dim
        self.num_classes = num_classes

        self.patch_dim = patch_size * patch_size * channels
        self.n_patches = (img_size // patch_size) ** 2

        # Bottleneck patch embedding: patch_dim -> hidden_dim
        # Key JiT result: this works even when patch_dim > hidden_dim
        self.patch_embed = nn.Linear(self.patch_dim, hidden_dim)

        # Learnable positional embedding
        self.pos_emb = nn.Parameter(torch.zeros(1, self.n_patches, hidden_dim))
        nn.init.normal_(self.pos_emb, std=0.02)

        # Time embedding: scalar t -> (B, time_emb_dim)
        self.time_emb = nn.Sequential(
            SinusoidalPosEmb(time_emb_dim),
            nn.Linear(time_emb_dim, time_emb_dim * 2),
            nn.SiLU(),
            nn.Linear(time_emb_dim * 2, time_emb_dim),
        )

        # Label embedding: class index -> (B, time_emb_dim)
        # num_classes + 1 = null token for Classifier-Free Guidance
        self.label_emb = nn.Embedding(num_classes + 1, time_emb_dim)

        # Fuse time + label into a single conditioning vector
        self.cond_mlp = nn.Sequential(
            nn.Linear(time_emb_dim * 2, time_emb_dim * 2),
            nn.SiLU(),
            nn.Linear(time_emb_dim * 2, time_emb_dim),
        )

        # Transformer blocks
        self.blocks = nn.ModuleList([
            TransformerBlock(hidden_dim, num_heads, time_emb_dim, mlp_ratio)
            for _ in range(depth)
        ])

        # Output: predict clean image patches (x-prediction)
        self.norm_out = nn.LayerNorm(hidden_dim)
        self.head     = nn.Linear(hidden_dim, self.patch_dim)

    # ------ patch utilities ------

    def patchify(self, x):
        """(B, C, H, W) -> (B, n_patches, patch_dim)"""
        B, C, H, W = x.shape
        p = self.patch_size
        x = x.reshape(B, C, H // p, p, W // p, p)  # (B, C, nh, p, nw, p)
        x = x.permute(0, 2, 4, 1, 3, 5)             # (B, nh, nw, C, p, p)
        return x.reshape(B, self.n_patches, self.patch_dim)

    def unpatchify(self, x):
        """(B, n_patches, patch_dim) -> (B, C, H, W)"""
        B, N, _ = x.shape
        p  = self.patch_size
        nh = nw = int(N ** 0.5)
        x = x.reshape(B, nh, nw, self.channels, p, p)  # (B, nh, nw, C, p, p)
        x = x.permute(0, 3, 1, 4, 2, 5)                # (B, C, nh, p, nw, p)
        return x.reshape(B, self.channels, nh * p, nw * p)

    # ------ forward pass ------

    def forward(self, z, t, labels, cfg_dropout=0.0):
        """
        Args:
            z          : noisy image  (B, C, H, W)
            t          : noise level  (B,) in [0, 1]
            labels     : class labels (B,) in [0, num_classes-1]
            cfg_dropout: label drop probability during training
        Returns:
            x_pred: predicted clean image (B, C, H, W)  <- x-prediction
        """
        B = z.shape[0]

        # CFG label dropout: replace label with null token
        if cfg_dropout > 0 and self.training:
            drop_mask       = torch.rand(B, device=z.device) < cfg_dropout
            labels          = labels.clone()
            labels[drop_mask] = self.num_classes  # null token

        # Patch embed + positional encoding
        patches = self.patchify(z)                     # (B, N, patch_dim)
        x = self.patch_embed(patches) + self.pos_emb   # (B, N, hidden_dim)

        # Conditioning: time + label
        t_emb = self.time_emb(t)                       # (B, time_emb_dim)
        l_emb = self.label_emb(labels)                 # (B, time_emb_dim)
        cond  = self.cond_mlp(torch.cat([t_emb, l_emb], dim=-1))  # (B, time_emb_dim)

        # Transformer blocks
        for block in self.blocks:
            x = block(x, cond)

        # x-prediction head
        x = self.norm_out(x)
        x = self.head(x)        # (B, N, patch_dim)

        return self.unpatchify(x)  # (B, C, H, W)

In [ ]:
# ============================================================
# Training objective: x-prediction with v-loss weighting
# ============================================================
#
# Flow matching linear path (same convention as flow_matching.ipynb):
#   z_t = (1-t)*x + t*eps    t=0: clean data,  t=1: pure noise
#
# x-prediction:
#   network output:  x_pred = JiT(z_t, t, label)
#
# v-loss (loss computed in velocity space):
#   v_target = (z_t - x) / t  =  eps - x
#   v_pred   = (z_t - x_pred) / t
#   loss     = ||v_pred - v_target||^2
#            = ||(x - x_pred) / t||^2       <- re-weighted by 1/t^2
#
# Why v-loss over plain MSE(x_pred, x)?
#   The 1/t^2 factor upweights small-t steps (near clean data),
#   forcing the model to be precise in the final denoising stage.

def train_step(x, labels, model, optimizer, cfg_dropout=0.1, t_min=0.02):
    B, device = x.shape[0], x.device

    # Sample time t in [t_min, 1]
    t  = torch.rand(B, device=device) * (1.0 - t_min) + t_min  # (B,)
    t4 = t[:, None, None, None]  # broadcast to image shape

    # Forward process
    eps = torch.randn_like(x)
    z_t = (1 - t4) * x + t4 * eps

    # x-prediction
    x_pred = model(z_t, t, labels, cfg_dropout=cfg_dropout)

    # v-loss: ||(x - x_pred) / t||^2
    loss = ((x - x_pred) / t4).pow(2).mean()

    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()

    return loss.item()


# ---- EMA (from mean_flow.ipynb) --------------------------------
def init_ema(net, net_ema, ema_decay):
    for p, p_ema in zip(net.parameters(), net_ema.parameters()):
        p_ema.data.copy_(p.data)
        p_ema.requires_grad = False
    net_ema.ema_decay = ema_decay
    return net_ema


def update_ema(net, net_ema):
    decay = net_ema.ema_decay
    with torch.no_grad():
        for p, p_ema in zip(net.parameters(), net_ema.parameters()):
            p_ema.data.mul_(decay).add_(p.data, alpha=1.0 - decay)

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")

# ---- Model config ----
# patch_size=7  => patch_dim=49,  n_patches=16  (28x28 image)
# hidden_dim=128 => mild bottleneck (49->128)
# Try patch_size=14 (patch_dim=196) to see sharper bottleneck effect
cfg = dict(
    img_size     = 28,
    patch_size   = 7,
    channels     = 1,
    hidden_dim   = 128,
    depth        = 6,
    num_heads    = 4,
    num_classes  = 10,
    time_emb_dim = 128,
)

model     = JiT(**cfg).to(device)
model_ema = JiT(**cfg).to(device)
init_ema(model, model_ema, ema_decay=0.9999)

n_params = sum(p.numel() for p in model.parameters())
print(f"Parameters : {n_params:,}")
print(f"patch_dim  : {model.patch_dim}  (= {model.patch_size}x{model.patch_size}x{model.channels})")
print(f"hidden_dim : {model.hidden_dim}")
print(f"n_patches  : {model.n_patches}  ({int(model.n_patches**0.5)}x{int(model.n_patches**0.5)} grid)")
print(f"Bottleneck : patch_dim={model.patch_dim} -> hidden_dim={model.hidden_dim}")

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.01)

# ---- Training loop ----
n_epochs = 50
losses   = []

for epoch in (pbar := tqdm(range(n_epochs), ncols=100)):
    model.train()
    epoch_loss = []
    for x, labels in train_loader:
        x, labels = x.to(device), labels.to(device)
        loss = train_step(x, labels, model, optimizer, cfg_dropout=0.1)
        epoch_loss.append(loss)
        update_ema(model, model_ema)
    mean_loss = np.mean(epoch_loss)
    losses.append(mean_loss)
    pbar.set_description(f"loss = {mean_loss:.4f}")

# ---- Loss curve ----
losses_np = np.array(losses)
pad       = min(5, len(losses_np) // 4)
losses_sm = sliding_window_view(np.pad(losses_np, (pad, pad), mode='reflect'), 2*pad+1).mean(axis=1)

with sns.axes_style("darkgrid"):
    fig, ax = plt.subplots(1, 1, figsize=(6, 3), dpi=150)
    ax.semilogy(losses_np, linewidth=0.8, alpha=0.5, label='raw')
    ax.semilogy(losses_sm, linewidth=1.5,             label='smoothed')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('v-loss')
    ax.set_title('JiT Training Loss  (x-prediction + v-loss weighting)')
    ax.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
# ============================================================
# Sampling: Euler ODE with Classifier-Free Guidance
# ============================================================
#
# Reverse flow: integrate from t=1 (noise) -> t=t_min (clean)
#   velocity field: v_theta = (z_t - x_pred) / t
#   Euler step:     z_{t+dt} = z_t + v * dt   (dt < 0, moving toward data)
#
# CFG in x-space (as used in JiT):
#   x_pred = x_uncond + cfg_scale * (x_cond - x_uncond)
#   Higher cfg_scale => more class-faithful, less diverse

@torch.no_grad()
def sample(model, n_samples, labels, n_steps=100, cfg_scale=3.0, t_min=0.02):
    device = next(model.parameters()).device
    model.eval()

    # Start from pure noise at t=1
    z = torch.randn(n_samples, model.channels, model.img_size, model.img_size, device=device)
    labels      = labels.to(device)
    null_labels = torch.full_like(labels, model.num_classes)  # unconditional

    # Time schedule: 1.0 -> t_min
    t_seq = torch.linspace(1.0, t_min, n_steps + 1, device=device)

    for i in range(n_steps):
        t     = t_seq[i]
        t_nxt = t_seq[i + 1]
        dt    = t_nxt - t    # negative (moving toward clean)

        t_batch = t.expand(n_samples)

        # x-prediction (conditional and unconditional)
        x_cond   = model(z, t_batch, labels)
        x_uncond = model(z, t_batch, null_labels)

        # CFG in x-space
        x_pred = x_uncond + cfg_scale * (x_cond - x_uncond)
        x_pred = x_pred.clamp(-1, 1)

        # Euler step via velocity field
        v = (z - x_pred) / t
        z = z + v * dt

    return z.clamp(-1, 1).cpu()

In [ ]:
model_ema.eval()

# ---- Generate 10 samples per class (10x10 grid) ----
all_samples = []
for cls in range(10):
    labels  = torch.full((10,), cls, dtype=torch.long)
    samples = sample(model_ema, 10, labels, n_steps=100, cfg_scale=3.0)
    all_samples.append(samples)

fig, axes = plt.subplots(10, 10, figsize=(12, 12))
for cls in range(10):
    for j in range(10):
        ax  = axes[cls, j]
        img = all_samples[cls][j].squeeze()
        ax.imshow(img, cmap='gray', vmin=-1, vmax=1)
        ax.axis('off')
        if j == 0:
            ax.set_ylabel(str(cls), rotation=0, labelpad=12, fontsize=10)
plt.suptitle('JiT Generated MNIST Digits\n(rows = class label, cfg_scale=3.0)', fontsize=13)
plt.tight_layout()
plt.show()

# ---- Effect of CFG scale ----
fig, axes = plt.subplots(3, 10, figsize=(14, 4))
for row, cfg_scale in enumerate([1.0, 3.0, 7.0]):
    labels  = torch.arange(10, dtype=torch.long)
    samples = sample(model_ema, 10, labels, n_steps=100, cfg_scale=cfg_scale)
    for j in range(10):
        ax = axes[row, j]
        ax.imshow(samples[j].squeeze(), cmap='gray', vmin=-1, vmax=1)
        ax.axis('off')
        if j == 0:
            ax.set_ylabel(f'cfg={cfg_scale}', fontsize=8)
plt.suptitle('Effect of CFG Scale  (higher = more class-faithful, less diverse)', fontsize=12)
plt.tight_layout()
plt.show()